In [1]:
import numpy as np

import jax.numpy as jnp
from jax import jit, vmap, grad
import jax.random

import scipy.optimize
import matplotlib.pyplot as plt

import emcee
import corner
#import tensorflow_probability.substrates.jax as tfp
#from bayesian_stats_course_tools.plotting import plot_contours

import numpyro.distributions as dist
import numpyro
from numpyro import infer


from numpyro.infer import NUTS, MCMC

C:\Users\L-Vis\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Exercise 1

For the linear model $y_i \sim N(mx_i + b, \sigma_i)$ and data from `../lectures/data/linear_fits/data_0.txt` do the following:

1. Analytically calculate the FIM using either the Hessian definition or the grad definition.
2. Write a function that calculates this based on the $x$-values and $\sigma$.
3. Implement the likelihood in JAX and use `jax.hessian` or `jax.grad` to calculate the FIM and compare to (2). If you use `jax.hessian`, check that properties of the analytic FIM propagate here (e.g. whether it is independent of certain variables).

# Exercise 2: Hierarchical models and probabilistic programming

We have test scores $y_{ij}$ for schools across several cities, where $j$ labels the city and $i$ labels the school within that city.
Schools in the same city share a mean test score $\theta_j$, and the city means are themselves drawn from a country-wide Gaussian:

$$y_{ij} \sim \mathcal N_y(\theta_j, \sigma_{ij}^2)$$ 
$$\theta_j \sim \mathcal N_\theta(\mu, \tau^2)$$
$$\mu \sim \mathcal U(0, 100)$$
$$\log_{10} \tau \sim \mathcal U(-2, 2)$$

where we have used a Jeffreys prior on $\tau$.

The data are in `lectures/data/fake_school_data.txt`: 20 cities with 10 schools each.
The file has 40 columns that alternate between the scores of the schools in a city and the variances $\sigma_{ij}^2$ of those scores
(column 1: scores in city 1, column 2: variances in city 1, column 3: scores in city 2, ...).

1. Implement this model in `numpyro`.
2. Sample the posterior of $\mu$, $\tau$ and the $\theta_j$ with NUTS.
3. Compare the posterior means of the $\theta_j$ with the per-city average scores. What does the hierarchical model do to them?

In [2]:
# Notation:
#   n_city             number of cities (groups), here 20
#   n_school_per_city  number of schools per city, here 10
school_table = np.loadtxt("../lectures/data/fake_school_data.txt")  # (n_school_per_city, 2 * n_city)

# Columns alternate (scores, variances) per city; transpose so cities are rows
scores = school_table[:, 0::2].T           # (n_city, n_school_per_city)
score_variances = school_table[:, 1::2].T  # (n_city, n_school_per_city)
score_sigmas = np.sqrt(score_variances)    # (n_city, n_school_per_city)

n_city, n_school_per_city = scores.shape
print(f"{n_city} cities with {n_school_per_city} schools each")

num_samples=10_000

20 cities with 10 schools each


In [3]:
## TEILAUFGABE 1


# numpyro.sample() erzeugt im probabilistischen Modell die entsprechenden Zufallsvariablen!

def model(scores, score_sigmas): #Funktion von NumPyro mit def model(,): starten

    #### Hyperprior
        mu = numpyro.sample("mu", dist.Uniform(0, 100))

        log_tau = numpyro.sample("log_tau", dist.Uniform(-2, 2))
        tau = 10**(log_tau)




    #### Hierarchical Distribution on city level
        theta = numpyro.sample("theta", dist.Normal(mu, tau).expand([n_city]))
        #Zwei Sachen hier wichtig: statt dist.Normal(mu, tau**2) sollte man mu, tau verwenden (korrekter Input von dist.Normal)

        # statt dist.Normal(mu, tau) sollte man dist.Normal(mu, tau).expand([n_city] verwenden! 
        # .expand([n_city]) sagt NumPyro, dass wir nicht nur eine Zufallsvariable, sondern n_city Zufallsvariablen mit derselben Verteilung benötigen.
        # mit nur dist.normal in NumPyro erzeugen wir genau eine Normalverteilung theta ~ N(theta, tau). Mit .expand([n_city]) erzeugen wir 20 solcher theta_i



     #Likelihood
        y = numpyro.sample(
            "y",
            dist.Normal(theta[:, None], score_sigmas),
            obs=scores                                  #obs=scores sehr wichtig! Sagt das Messdaten bereits beobachtet und wo sie sind (hier in scores)
        )
    #theta[:, None] ist sehr wichtig damit Stadtmittelwerte (20) ---> (20,1) aussehen!
    #(20, 1) Stadtmittelwerte + (20, 10) Unsicherheiten --> (20, 10) 





#Hier ist Funktion beendet wegen Indentation fertig!

In [14]:
## TEILAUFGABE 2



# Wähl den NUTS Sampler
kernel = NUTS(model)

# Konfigurier MCMC Simulation
mcmc = MCMC(kernel, num_warmup=1000, num_samples=2000)



# Wähl einfach einen Seed damit es reproduzierbar ist. Hier wurde 0 als Seed gewählt.


#Observed data = scores, scores_sigmas
#mcmc.run(jax.random.key(0), data) --> data = wie das was im Model gewählt wurde. Also bei uns war
# def model(scores, score_sigmas): deswegen muss bei data scores, scores_sigma stehen!

mcmc.run(jax.random.key(0), scores, score_sigmas)



samples = mcmc.get_samples() #Ich bekomme damit ein Dictionary von meinen Posterior Samples
tau_samples = 10 ** samples["log_tau"]

sample: 100%|██████████| 3000/3000 [00:04<00:00, 632.48it/s, 7 steps of size 5.37e-01. acc. prob=0.89] 


In [11]:
#Workaround (musste eingeführt werden)

import importlib

dist.distribution = importlib.import_module(
    "numpyro.distributions.distribution"
)

print("Distribution:", hasattr(dist, "distribution"))

Distribution: True


In [ ]:
## TEILAUFGABE 3


# per-city average scores sind naive Mittelwerte einfach arithmetisches Mittel pro Stadt ausrechnen
# Pro Stadt die Scores aller Schulen der Stadt zusammenaddieren und dann durch Anzahl Schulen teilen:



naive_average = np.mean(scores, axis = 1) #axis = 1 bedeutet, dass der Mittelwert über die Spalten jeder Zeile berechnet wird.
print(naive_average)

#axis = 0 --> Zeilen wurden zusammengefasst
#axis = 1 --> Spalten wurden zusammengefasst (macht hier viel mehr Sinn, da eine Stadt eine Spalte ist)





[63.05077 65.90628 53.2367  88.34674 75.02609 63.39699 66.47234 78.89028
 66.96629 72.79725 72.67554 72.75892 68.73294 66.74238 72.30203 64.50885
 64.96107 75.24918 68.35958 56.91841]


In [17]:
mcmc.print_summary()


                mean       std    median      5.0%     95.0%     n_eff     r_hat
   log_tau      0.88      0.08      0.87      0.74      1.00   2278.58      1.00
        mu     69.29      1.90     69.29     66.46     72.57   2723.19      1.00
  theta[0]     62.54      1.96     62.57     59.17     65.60   3590.63      1.00
  theta[1]     67.28      2.37     67.32     63.21     70.92   3484.42      1.00
  theta[2]     55.97      2.16     56.01     52.52     59.71   3362.90      1.00
  theta[3]     86.51      2.21     86.58     83.06     90.11   4333.31      1.00
  theta[4]     76.06      2.04     76.07     72.67     79.42   3643.72      1.00
  theta[5]     64.99      2.17     64.99     61.30     68.36   3303.64      1.00
  theta[6]     68.10      2.12     68.13     64.74     71.80   3819.63      1.00
  theta[7]     75.01      2.47     74.99     71.16     79.39   3807.51      1.00
  theta[8]     68.05      2.19     68.05     64.27     71.40   3672.21      1.00
  theta[9]     71.92      1

## Vergleich von naive average vs Bayesian Hierarchical


Der naive Mittelwert berücksichtigt nur die 10 Schulen einer Stadt. Die Bayesian-Schätzung berücksichtigt zusätzlich, dass die Städte einer gemeinsamen Verteilung entstammen.


Das Bayesian Modell schätzt übergeordneten Mittelwert $\mu = 69.29 \pm 1.90$. 


Bayesian Modell zieht extreme Städte tendenziell gegen Mittelwert $\mu = 69.29$. Also Städte wo stark über 69.29 liegen werden bisschen runtergezogen, Städte welche stark unter 69.29 liegen werden bisschen hochgezogen. Das ist *Shrinkage* das ist zu erwarten von *partial pooling*.


